# DKT Next-Item — Colab

Notebook pronto para testar o modelo **DKT Next-Item** no Google Colab.

Fluxo sugerido:
1. Executar setup do ambiente.
2. Montar Google Drive.
3. Copiar os dados para `data/`.
4. Executar o pipeline de dados se os artefatos ainda não existirem.
5. Rodar o treino deste modelo.


In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/GuilhermeDesoler/ai-core.git"
REPO_DIR = Path("/content/ai-core")

if IN_COLAB:
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd', '/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    print("Rodando fora do Colab.")
    REPO_DIR = Path.cwd()

os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
print("Repo:", REPO_DIR)
print("PYTHONPATH:", os.environ["PYTHONPATH"])


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/ai-core-data")
    print("Drive montado. Ajuste DRIVE_DATA_ROOT conforme sua estrutura.")
except Exception as e:
    print("Mount do Drive não disponível neste ambiente:", e)


In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = REPO_DIR
DATA_TARGET = PROJECT_ROOT / "data"
DATA_TARGET.mkdir(parents=True, exist_ok=True)

# Ajuste estes caminhos conforme onde seus dados estão no Drive:
SOURCE_RAW_DIR = Path("/content/drive/MyDrive/ai-core-data/raw")
SOURCE_PROCESSED_DIR = Path("/content/drive/MyDrive/ai-core-data/processed")

# Descomente quando quiser copiar:
# shutil.copytree(SOURCE_RAW_DIR, DATA_TARGET / "raw", dirs_exist_ok=True)
# shutil.copytree(SOURCE_PROCESSED_DIR, DATA_TARGET / "processed", dirs_exist_ok=True)

print("Pasta data alvo:", DATA_TARGET)
print("Verifique se data/raw e/ou data/processed já existem antes de prosseguir.")


In [ ]:
from pathlib import Path

checks = [
    Path("data/raw/answers.json"),
    Path("data/processed/dataset/answers_prepared.csv"),
    Path("data/processed/sequences/user_sequences.json"),
    Path("data/processed/mappings/skill_to_h2.json"),
    Path("data/processed/mappings/skill_to_h3.json"),
]

for p in checks:
    print(f"{p}: {p.exists()}")


In [ ]:
# Rode esta célula se você ainda precisa gerar answers_prepared.csv,
# user_sequences.json e os mapeamentos H2/H3.
get_ipython().system('python run_data_pipeline.py')


In [ ]:
# Opcional e recomendado para testes com sequências por sessão.
get_ipython().system('python scripts/build_user_session_sequences.py')


In [ ]:
# Treino do modelo DKT Next-Item
get_ipython().system('PYTHONPATH=./src python scripts/train_eval_dkt_next_item.py')


In [ ]:
from pathlib import Path
import json

artifacts_dir = Path("artifacts")
print("Artifacts existe?", artifacts_dir.exists())

for p in sorted(artifacts_dir.rglob("metrics.json"))[-10:]:
    print(p)
    try:
        print(json.loads(p.read_text()))
    except Exception as e:
        print("Não foi possível ler metrics.json:", e)
    print("-" * 80)
